In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/09 19:10:54 WARN Utils: Your hostname, ZenbookUX3402ZA, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/09 19:10:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 19:11:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
yellow_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("passenger_count", types.IntegerType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("RatecodeID", types.IntegerType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("payment_type", types.IntegerType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

# Source file: "yellow_tripdata_2025-11.parquet" downloaded from https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

In [3]:
df_yellow = spark.read.parquet('data/pq/yellow/2025/11/yellow_tripdata_2025-11.parquet')

In [4]:
df_yellow.show()

[Stage 2:============================================>              (3 + 1) / 4]

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

# Write new partitioned parquet files

In [5]:
df_yellow.repartition(4).write.parquet('data/pq/yellow/2025/11/repartitioned/', mode='overwrite')

In [6]:
df_yellow.count()

4181444

In [7]:
from pyspark.sql import functions as F

# Filter the dataframe record by date

In [8]:
df_yellow_15_nov = df_yellow.filter(F.to_date(df_yellow.tpep_pickup_datetime) == "2025-11-15").count()

In [10]:
df_yellow_15_nov

162604

# Add a new column with trip duration in hours

In [11]:
df_with_duration = df_yellow.withColumn(
    "duration_hours",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 3600
)

In [12]:
max_trip_duration = df_with_duration.select(F.max("duration_hours")).collect()[0][0]

In [13]:
max_trip_duration

90.64666666666666

In [15]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-09 19:42:15--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.165.247.187, 3.165.247.163, 3.165.247.79, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.165.247.187|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0.01s   

2026-03-09 19:42:15 (1.07 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [20]:
df_zones = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv('taxi_zone_lookup.csv')

In [21]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [22]:
df_joined = df_yellow.join(df_zones, df_yellow.PULocationID == df_zones.LocationID)

In [23]:
least_frequent_zone = df_joined.groupBy("Zone") \
    .count() \
    .orderBy("count", ascending=True)

In [24]:
least_frequent_zone.show()

[Stage 21:==========================================>             (12 + 4) / 16]

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|Governor's Island...|    1|
|Eltingville/Annad...|    1|
|       Arden Heights|    1|
|       Port Richmond|    3|
|       Rikers Island|    4|
|   Rossville/Woodrow|    4|
| Green-Wood Cemetery|    4|
|         Great Kills|    4|
|         Jamaica Bay|    5|
|         Westerleigh|   12|
|        Crotona Park|   14|
|             Oakwood|   14|
|New Dorp/Midland ...|   14|
|       West Brighton|   14|
|       Willets Point|   15|
|Breezy Point/Fort...|   16|
|Saint George/New ...|   17|
|       Broad Channel|   18|
|     Mariners Harbor|   21|
|Heartland Village...|   22|
+--------------------+-----+
only showing top 20 rows
